In [0]:
from delta.tables import DeltaTable
from pyspark.sql import functions as F
from datetime import datetime

# ==========================================
# CAPTURA Y SANITIZACIÓN DE PARÁMETROS
# ==========================================
dbutils.widgets.text("fecha_carga", "")
dbutils.widgets.text("modo", "incremental")

param_fecha = dbutils.widgets.get("fecha_carga").strip()
modo = dbutils.widgets.get("modo").strip().lower()

# ==========================================
# ESCRITURA EN SILVER (Variante B: MERGE vs OVERWRITE)
# ==========================================
target_table = "bootcamp.silver.propiedades"
table_exists = spark.catalog.tableExists(target_table)

if modo == "full" or not table_exists:
    print(f"Ejecutando INSERT OVERWRITE en {target_table}...")
    (
        df_silver_source.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(target_table)
    )

elif modo == "incremental":
    print(f"Ejecutando MERGE (UPSERT) en {target_table}...")
    target_delta = DeltaTable.forName(spark, target_table)

    # Condición de clave única (PK)
    merge_condition = "target.id_evento = source.id_evento"

    (
        target_delta.alias("target")
        .merge(
            df_silver_source.alias("source"),
            merge_condition
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )

else:
    raise ValueError(f"Modo inválido: '{modo}'. Debe ser 'full' o 'incremental'.")